# Churn de suscripciones: Entorno

Comprueba el Python activo y los imports. Si instala dependencias, reinicia el kernel y repite.

**Estado:** ejecutado el 6 de octubre de 2026; resultados exploratorios revisados en README y docs/ejecuciones.md. Consulta ../docs/metodologia.md.


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("subscription_churn")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from subscription_churn.alertas import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
import importlib
import importlib.util
import subprocess
import json
from datetime import datetime, timezone
modulos = ["numpy", "pandas", "sklearn", "pyarrow", "matplotlib", "joblib", "IPython", "nbclient", "nbformat"]
faltantes = [m for m in modulos if importlib.util.find_spec(m) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], cwd=PROJECT_ROOT)
    raise RuntimeError("Dependencias instaladas. Reinicia el kernel y repite 00.")
versiones = {m: getattr(importlib.import_module(m), "__version__", "no declarada") for m in modulos}
check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
from subscription_churn.flujo import guardar_json
guardar_json(PROJECT_ROOT / "reports/entorno/environment.json", {"fecha_utc": datetime.now(timezone.utc).isoformat(), "python": sys.executable, "versiones": versiones, "pip_check_ok": check.returncode == 0, "pip_check": check.stdout + check.stderr})
display(versiones)
if check.returncode:
    raise RuntimeError("pip check detecto incompatibilidades; revisa reports/entorno/environment.json.")


{'numpy': '2.3.5',
 'pandas': '2.3.3',
 'sklearn': '1.9.1',
 'pyarrow': '25.0.1',
 'matplotlib': '3.11.2',
 'joblib': '1.6.0',
 'IPython': '9.8.0',
 'nbclient': '0.11.0',
 'nbformat': '5.11.1'}

## Alertas de la ejecucion

Avisos agrupados con descripcion generica, acciones sugeridas y detalle original. Los errores de celda no se ocultan.

In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/00_configuracion_entorno.json")